In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("nudratabbas/software-developer-salary-prediction-dataset")
print(path)

c:\Nyeh\Code\Python\Machine_Learning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C:\Users\Administrator\.cache\kagglehub\datasets\nudratabbas\software-developer-salary-prediction-dataset\versions\1


In [12]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error as MSE
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor 

In [3]:
data = pd.read_csv(os.path.join(path, os.listdir(path)[1]))
data.head()

,experience,country,education,languages,frameworks,company_size,salary_usd
0,10,Singapore,Bachelors,"JavaScript, Go","Spring, Express",1-10,68491
1,13,USA,Some College,"PHP, Java","Flask, Ruby on Rails",51-200,120045
2,28,India,Some College,"JavaScript, C#","Vue, Spring",51-200,87811
3,16,Brazil,Bachelors,"Ruby, JavaScript","React, Spring",201-1000,99426
4,20,Singapore,PhD,"Java, Java","Laravel, Laravel",201-1000,108251


In [4]:
# print(len(data.select_dtypes(include="str").columns))

# for i in data.select_dtypes(include="object").columns:
#     print(i.upper())
#     print(f"Number of unique elements {data[i].nunique()}")    
#     print(f"unique elements: {data[i].unique()}\n",)    

data["company_size"].unique()
# data.isna().sum()

<StringArray>
['1-10', '51-200', '201-1000', '1001-5000', '5000+', '11-50']
Length: 6, dtype: str

In [6]:

mlb_lang=  MultiLabelBinarizer()
mlb_frame = MultiLabelBinarizer()

lang_encoded = pd.DataFrame(
    mlb_lang.fit_transform(data["languages"].str.split(", ")),
    columns=mlb_lang.classes_,
    index = data.index
)
frame_encoded = pd.DataFrame(
    mlb_frame.fit_transform(data["frameworks"].str.split(", ")),
                            columns=mlb_frame.classes_,
                            index = data.index
)

data.drop(columns=["languages","frameworks"], inplace=True)
data = pd.concat([data, lang_encoded, frame_encoded], axis=1)

size_order = {"1-10": 1, "11-50": 2 , "51-200": 3, "201-1000": 4, "1001-5000": 5, "5000+" : 6}
data["company_size"] = data["company_size"].map(size_order)

edu_order = {"High School": 1, "Some College": 2, "Bachelors": 3, "Masters": 4, "PhD": 5}
data["education"] = data["education"].map(edu_order)


In [10]:
data_dummified = pd.get_dummies(data, columns=["country"])
# data_dummified.drop(columns=["languages", "frameworks"], inplace=True)
data_dummified.isna().sum()

experience           0
education            0
company_size         0
salary_usd           0
C#                   0
C++                  0
Go                   0
Java                 0
JavaScript           0
PHP                  0
Python               0
Ruby                 0
Rust                 0
Swift                0
ASP.NET              0
Angular              0
Django               0
Express              0
Flask                0
Laravel              0
React                0
Ruby on Rails        0
Spring               0
Vue                  0
country_Australia    0
country_Brazil       0
country_Canada       0
country_France       0
country_Germany      0
country_India        0
country_Japan        0
country_Singapore    0
country_UK           0
country_USA          0
dtype: int64

In [16]:
linreg = LinearRegression()

X = data_dummified.drop("salary_usd", axis=1).values
y = data_dummified["salary_usd"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

linreg.fit(X_train_scaled, y_train)
y_pred = linreg.predict(X_test_scaled)

score = linreg.score(X_test_scaled, y_test)
mse = MSE(y_test, y_pred)
root_MSE = np.sqrt(mse)

train_score = linreg.score(X_train, y_train)
test_score = linreg.score(X_test, y_test)
print(f"Train R²: {train_score:.4f}")
print(f"Test R²:  {test_score:.4f}")

print(f"SCORE: {score}")
print(f"MSE: {mse}")
print(f"RMSE: {root_MSE}")


Train R²: -321.9630
Test R²:  -312.6767
SCORE: 0.8701200358669584
MSE: 289688095.30404896
RMSE: 17020.226065010094
